# SenSia — Analisis Sentimen Komentar Saham Indonesia

**Nama:** _(isi nama)_  
**Mata kuliah:** _(isi mata kuliah)_  
**Repo:** https://github.com/_(isi username)_/Sensia

Pipeline: dataset berlabel dari X dan Threads -> cleaning teks (`sensia.preprocess`) -> tokenisasi -> Bi-LSTM 3 kelas (negatif/netral/positif) -> evaluasi -> artefak model untuk web app Flask.

Kontrak teks hidup di paket `sensia/` supaya notebook dan web app memakai cleaning yang sama.


In [1]:
import json, pathlib, random
import numpy as np, pandas as pd, tensorflow as tf
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sensia.config import LABELS, MAX_LEN, SEED, VOCAB_SIZE
from sensia.preprocess import basic_clean, clean_text, normalize_slang
from sensia.dataset import validate_labeled, dedupe_labeled

random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print("tensorflow", tf.__version__, "| labels", LABELS)

I0000 00:00:1795196150.410691   55064 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1795196151.478082   55064 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


I0000 00:00:1795196154.702390   55064 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


tensorflow 2.21.0 | labels ['negatif', 'netral', 'positif']


In [2]:
df = dedupe_labeled(validate_labeled(pd.read_excel("data/labeled/sensia_labeled.xlsx")))
print(df.shape)
df["label"].value_counts()

(1108, 5)


label
netral     812
positif    175
negatif    121
Name: count, dtype: int64

In [3]:
fig, ax = plt.subplots(1, 3, figsize=(15, 3.5))
df["label"].value_counts().reindex(LABELS).plot.bar(ax=ax[0], color=["#c0392b", "#7f8c8d", "#27ae60"], title="label")
df["platform"].value_counts().plot.bar(ax=ax[1], title="platform")
df["text"].str.split().str.len().plot.hist(ax=ax[2], bins=30, title="panjang teks (kata)")
plt.tight_layout(); plt.show()

In [4]:
df["clean"] = df["text"].map(clean_text)
assert (df["clean"].str.len() > 0).all(), "ada teks yang kosong setelah cleaning"
df[["text", "clean"]].head()

,text,clean
0,IHSG anjlok Harga emas turun Rupiah melemah te...,ihsg anjlok harga emas turun rupiah melemah te...
1,"Mulai 2 November 2026, saham HSC dikeluarkan d...","mulai 2 november 2026, saham hsc dikeluarkan d..."
2,"Yg pakar tolong, apa penyebab IHSG anjlok para...","yang pakar tolong, apa penyebab ihsg anjlok pa..."
3,"Hey, hey.. Fakkk! Prabowo pidato: IHSG anjlok ...","hey, hey.. fakkk prabowo pidato ihsg anjlok ad..."
4,"Sempat Anjlok di pembukaan, IHSG Langsung Rebo...","sempat anjlok di pembukaan, ihsg langsung rebo..."


In [5]:
demo = pd.DataFrame({"basic_clean": df["text"].head(5).map(basic_clean), "clean_text": df["clean"].head(5)})
demo

,basic_clean,clean_text
0,ihsg anjlok harga emas turun rupiah melemah te...,ihsg anjlok harga emas turun rupiah melemah te...
1,"mulai 2 november 2026, saham hsc dikeluarkan d...","mulai 2 november 2026, saham hsc dikeluarkan d..."
2,"yg pakar tolong, apa penyebab ihsg anjlok para...","yang pakar tolong, apa penyebab ihsg anjlok pa..."
3,"hey, hey.. fakkk prabowo pidato ihsg anjlok ad...","hey, hey.. fakkk prabowo pidato ihsg anjlok ad..."
4,"sempat anjlok di pembukaan, ihsg langsung rebo...","sempat anjlok di pembukaan, ihsg langsung rebo..."


In [6]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(df, test_size=0.2, stratify=df["label"], random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df["label"], random_state=SEED)
y_train, y_val, y_test = (s["label"].map(LABELS.index).values for s in (train_df, val_df, test_df))
print(len(train_df), len(val_df), len(test_df))

886 111 111


In [7]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(train_df["clean"])

def to_seq(texts):
    return pad_sequences(tokenizer.texts_to_sequences(texts), maxlen=MAX_LEN, padding="post", truncating="post")

X_train, X_val, X_test = (to_seq(s["clean"]) for s in (train_df, val_df, test_df))
X_train.shape, X_val.shape, X_test.shape

((886, 100), (111, 100), (111, 100))

In [8]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout

model = Sequential([
    Embedding(VOCAB_SIZE, 128, input_length=MAX_LEN),
    Bidirectional(LSTM(64, dropout=0.2, recurrent_dropout=0.0)),
    Dropout(0.3),
    Dense(64, activation="relu"),
    Dense(3, activation="softmax"),
])
model.compile(loss="sparse_categorical_crossentropy", optimizer="adam", metrics=["accuracy"])
model.summary()

/home/mhmmdhdyt/Projects/Sensia/.venv/lib/python3.12/site-packages/keras/src/layers/core/embedding.py:123: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
E0000 00:00:1795196158.616337   55064 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [9]:
from tensorflow.keras.callbacks import EarlyStopping

hist = model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=15, batch_size=32,
                 callbacks=[EarlyStopping(patience=3, restore_best_weights=True)], verbose=2)
pd.DataFrame(hist.history)[["accuracy", "val_accuracy"]].plot(title="accuracy")
pd.DataFrame(hist.history)[["loss", "val_loss"]].plot(title="loss")
plt.show()

Epoch 1/15


28/28 - 9s - 320ms/step - accuracy: 0.7144 - loss: 0.8199 - val_accuracy: 0.7297 - val_loss: 0.7311


Epoch 2/15


28/28 - 2s - 80ms/step - accuracy: 0.7348 - loss: 0.7002 - val_accuracy: 0.7297 - val_loss: 0.6637


Epoch 3/15


28/28 - 2s - 79ms/step - accuracy: 0.7901 - loss: 0.5171 - val_accuracy: 0.7027 - val_loss: 0.7479


Epoch 4/15


28/28 - 2s - 80ms/step - accuracy: 0.8962 - loss: 0.2480 - val_accuracy: 0.7297 - val_loss: 1.0564


Epoch 5/15


28/28 - 2s - 85ms/step - accuracy: 0.9526 - loss: 0.1421 - val_accuracy: 0.7207 - val_loss: 1.2171


In [10]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

proba = model.predict(X_test)
y_pred = proba.argmax(axis=1)
print(classification_report(y_test, y_pred, target_names=LABELS, digits=3, zero_division=0))
print("confusion matrix (baris = benar, kolom = prediksi)")
print(confusion_matrix(y_test, y_pred, labels=range(3)))
print("accuracy", round(float(accuracy_score(y_test, y_pred)), 4),
      "| macro_f1", round(float(f1_score(y_test, y_pred, average="macro", zero_division=0)), 4))

majority = np.bincount(y_train).argmax()
print("baseline majority class:", LABELS[majority], round(float((y_test == majority).mean()), 4))

wrong = test_df.reset_index(drop=True).assign(pred=[LABELS[i] for i in y_pred])
wrong = wrong[wrong["pred"] != wrong["label"]]
print("\n10 contoh salah prediksi:")
for _, r in wrong.head(10).iterrows():
    print(f"- {r['text'][:120]!r} | benar={r['label']} prediksi={r['pred']}")

1/4 ━━━━━━━━━━━━━━━━━━━━ 1s 389ms/step

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 171ms/step

4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 178ms/step


              precision    recall  f1-score   support

     negatif      0.000     0.000     0.000        12
      netral      0.739     1.000     0.850        82
     positif      0.000     0.000     0.000        17

    accuracy                          0.739       111
   macro avg      0.246     0.333     0.283       111
weighted avg      0.546     0.739     0.628       111

confusion matrix (baris = benar, kolom = prediksi)
[[ 0 12  0]
 [ 0 82  0]
 [ 0 17  0]]
accuracy 0.7387 | macro_f1 0.2832
baseline majority class: netral 0.7387

10 contoh salah prediksi:
- 'Lebih nyerok tring kak, emas lagi turun hehehehe' | benar=negatif prediksi=netral
- "#Gerakaninfaqretailminusberjama'ah - MARI INFAQ BERSAMA-SAMA DENGAN MENYISIHKAN 2% DARI SELURUH MINUS KITA DI SAHAM - YU" | benar=negatif prediksi=netral
- 'ARB berjilid kak, fund gede dri msci antri mau out.. Tggu di 25 mgkn bru bisa.. Kepalang hold ja kak, prshaan msih runni' | benar=negatif prediksi=netral
- 'Klo account ritel kayaknya ar

In [11]:
pathlib.Path("model").mkdir(exist_ok=True)
model.save("model/sensia_bilstm.keras")
pathlib.Path("model/tokenizer.json").write_text(tokenizer.to_json())
pathlib.Path("model/label_map.json").write_text(json.dumps({str(i): l for i, l in enumerate(LABELS)}))
pathlib.Path("webapp").mkdir(exist_ok=True)
pathlib.Path("webapp/metrics.json").write_text(json.dumps({
    "accuracy": round(float(accuracy_score(y_test, y_pred)), 4),
    "macro_f1": round(float(f1_score(y_test, y_pred, average="macro", zero_division=0)), 4),
    "n_train": int(len(train_df)), "n_val": int(len(val_df)), "n_test": int(len(test_df)),
}))

def prediksi(teks):
    tokens = tokenizer.texts_to_sequences([clean_text(teks)])[0]
    if not tokens:
        return {"error": "teks tidak mengandung kata yang bisa dianalisis"}
    seq = pad_sequences([tokens], maxlen=MAX_LEN, padding="post", truncating="post")
    p = model.predict(seq, verbose=0)[0]
    return {"label": LABELS[int(p.argmax())], "confidence": float(p.max()),
            "scores": {l: round(float(x), 4) for l, x in zip(LABELS, p)}}

for s in ["ihsg anjlok parah, kapok main saham", "waktunya beli, bbca murah banget", "ihsg ditutup di 6812"]:
    print(s, "->", prediksi(s))

ihsg anjlok parah, kapok main saham -> {'label': 'netral', 'confidence': 0.8655627369880676, 'scores': {'negatif': 0.0406, 'netral': 0.8656, 'positif': 0.0939}}
waktunya beli, bbca murah banget -> {'label': 'netral', 'confidence': 0.8799909353256226, 'scores': {'negatif': 0.0323, 'netral': 0.88, 'positif': 0.0877}}
ihsg ditutup di 6812 -> {'label': 'netral', 'confidence': 0.924578070640564, 'scores': {'negatif': 0.0197, 'netral': 0.9246, 'positif': 0.0558}}
